# VaR Calculator - Historical Method

Build Value at Risk from scratch using historical returns.
- Gather past returns
- Sort them worst to best
- Find the percentile at your confidence level

## Setup: Imports

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import yfinance as yf

print('Imports loaded')

Imports loaded


## Download Real Data

In [ ]:
def download_price_data(ticker, days=252):
    """
    Download historical price data.
    Args:
        ticker: Stock ticker (e.g., 'SPY')
        days: Number of days of history (252 ~ 1 trading year)
    Returns:
        pd.Series of daily adjusted close prices
    """
    print(f'Downloading {ticker}...')
    try:
        data = yf.download(ticker, period=f'{days}d', progress=False)
        if isinstance(data.columns, pd.MultiIndex):
            return data.xs('Close', level=0, axis=1)[ticker]
        else:
            return data['Close']
    except Exception as e:
        print(f'Error downloading {ticker}: {e}')
        raise

def calculate_returns(prices):
    """Calculate daily returns from prices."""
    return prices.pct_change().dropna()

# Test
spy_prices = download_price_data('SPY', days=252)
print(f'Downloaded {len(spy_prices)} days of SPY data')
print(f'Date range: {spy_prices.index[0].date()} to {spy_prices.index[-1].date()}')

## Historical VaR Function

In [ ]:
def var_historical(returns, confidence=0.95):
    """
    Calculate VaR using historical method.
    
    Steps:
      1. Take past returns
      2. Sort from worst to best
      3. Find the cutoff at the confidence level
    
    Args:
        returns: Array or Series of historical daily returns
        confidence: Confidence level (0.95 = 95%, look at worst 5%)
    
    Returns:
        var_pct: VaR as a percentage (e.g., -0.025 means 2.5% loss)
    """
    percentile = (1 - confidence) * 100
    var_pct = np.percentile(returns, percentile)
    return var_pct

# Test on SPY
spy_returns = calculate_returns(spy_prices)
spy_var = var_historical(spy_returns, confidence=0.95)
print(f'SPY VaR (95%): {spy_var*100:.4f}%')

## Portfolio-Level VaR

In [ ]:
def var_portfolio_historical(tickers, weights, days=252, confidence=0.95):
    """
    Calculate VaR for a portfolio of multiple assets.
    
    Approach:
      1. Get historical prices for each ticker
      2. Calculate returns for each
      3. Build daily portfolio returns by weighting them
      4. Apply historical VaR
    
    Args:
        tickers: List of tickers (e.g., ['SPY', 'BND'])
        weights: List of portfolio weights (must sum to 1.0)
        days: Historical lookback period
        confidence: Confidence level
    
    Returns:
        dict with VaR and detailed breakdown
    """
    
    if not np.isclose(sum(weights), 1.0):
        raise ValueError(f'Weights must sum to 1.0, got {sum(weights)}')
    
    print(f'\n{"="*70}')
    print(f'PORTFOLIO VaR CALCULATION')
    print(f'{"="*70}')
    print(f'Tickers: {tickers}')
    print(f'Weights: {weights}')
    print(f'Lookback: {days} days')
    print(f'Confidence: {confidence*100}%')
    print(f'{"="*70}\n')
    
    # Step 1: Download prices and calculate returns for each asset
    all_returns = {}
    for ticker in tickers:
        prices = download_price_data(ticker, days=days)
        returns = calculate_returns(prices)
        all_returns[ticker] = returns
    
    # Step 2: Align returns (make sure all have same dates)
    returns_df = pd.DataFrame(all_returns)
    returns_df = returns_df.dropna()
    
    print(f'Data loaded. Using {len(returns_df)} trading days.\n')
    
    # Step 3: Calculate weighted portfolio returns
    portfolio_returns = (returns_df * weights).sum(axis=1)
    
    # Step 4: Calculate VaR
    var_pct = var_historical(portfolio_returns, confidence=confidence)
    
    # Collect statistics
    result = {
        'var_pct': var_pct,
        'mean_return_pct': portfolio_returns.mean() * 100,
        'std_dev': portfolio_returns.std() * 100,
        'min_return': portfolio_returns.min() * 100,
        'max_return': portfolio_returns.max() * 100,
        'returns_df': returns_df,
        'portfolio_returns': portfolio_returns,
        'individual_vars': {}
    }
    
    # Calculate individual VaRs for reference
    for ticker, weight in zip(tickers, weights):
        individual_var = var_historical(all_returns[ticker], confidence=confidence)
        result['individual_vars'][ticker] = individual_var
    
    return result

## Calculate 60/40 Portfolio

In [ ]:
# 60% stocks (SPY), 40% bonds (BND)
tickers = ['SPY', 'BND']
weights = [0.6, 0.4]

result = var_portfolio_historical(
    tickers=tickers,
    weights=weights,
    days=252,
    confidence=0.95
)

## Print Results

In [ ]:
def print_var_results(result, portfolio_value_usd=10_000_000, confidence=0.95):
    """Print VaR in a human-readable way."""
    
    var_pct = result['var_pct']
    var_dollar = var_pct * portfolio_value_usd
    
    print(f'\n{"="*70}')
    print(f'VaR RESULTS')
    print(f'{"="*70}\n')
    
    print(f'Portfolio Value:        ${portfolio_value_usd:,.0f}')
    print(f'Confidence Level:       {confidence*100}%')
    print(f'\n{"VaR (Daily):":.<40} {var_pct*100:>8.3f}%')
    print(f'{"VaR ($):":.<40} ${var_dollar:>15,.0f}')
    print(f'\nInterpretation:')
    print(f'  On 95% of trading days, portfolio loses at most ${abs(var_dollar):,.0f}')
    print(f'  On 5% of trading days (bad days), portfolio loses MORE than this.\n')
    
    print(f'Portfolio Statistics:')
    print(f'  Mean daily return:     {result["mean_return_pct"]:>8.4f}%')
    print(f'  Std deviation:         {result["std_dev"]:>8.4f}%')
    print(f'  Worst day:             {result["min_return"]:>8.3f}%')
    print(f'  Best day:              {result["max_return"]:>8.3f}%')
    
    print(f'\nIndividual Asset VaRs ({confidence*100}%):')
    for ticker, var in result['individual_vars'].items():
        print(f'  {ticker}:              {var*100:>8.3f}%')
    
    print(f'\n{"="*70}\n')

print_var_results(result, portfolio_value_usd=10_000_000, confidence=0.95)

## Experiment: Different Confidence Levels

In [ ]:
print(f'\n{"="*70}')
print(f'VaR AT DIFFERENT CONFIDENCE LEVELS')
print(f'{"="*70}\n')

for conf in [0.90, 0.95, 0.99]:
    var_pct = var_historical(result['portfolio_returns'], confidence=conf)
    var_dollar = var_pct * 10_000_000
    print(f'  {conf*100}% confidence: {var_pct*100:>7.3f}% = ${var_dollar:>15,.0f}')

print()